# Test inverses over signature
- DAJones Feb 2026

This is a test harness originally built to show that the Jones 6D algorithm works in all signatures, 
including degeneracy.
Naturally, this kind of testing was extended to include the Acus 6D algorithm from which the Jones algorithm was developed.
The `clifford` package includes a general purpose multivector inverse integrated into the multivector class.
This is accessed `iA = A.inv()`.
An improved general purpose multivector inverse has been included in the `clifford` baseline.
This is accessed `iA = A.shirokov_inverse()`.


The algorithms were ported to Python using the publically available multivector `clifford` package available through *conda* (the Anaconda package manager).  



In [1]:
import clifford as cf
import numpy as np
import warnings

from numba import NumbaDeprecationWarning
# Silence the noise
warnings.simplefilter('ignore', category=NumbaDeprecationWarning)

from jones_6D_inverse import jones_6D_inverse
from acus_6D_inverse import acus_6D_inverse
def hitzer_inverse(A): return A.hitzer_inverse() # dim < 6
# general purpose
def shirokov_inverse(A): return A.shirokov_inverse() # no dim restriction
def perwass_inverse(A): return A.leftLaInv() # no dim restriction


In [3]:

def run_tests(algorithm):
    print(f"\nTesting {algorithm.__name__}")
    # Header for the results table
    print(f"{'Signature (p,q,r)':<18} | {'Status':<10} | {'Avg Abs Error (MAE)':<20}")
    print("-" * 55)
    
    # Generate all 28 signatures for 6D
    signatures = [(p, q, 6-p-q) for p in range(7) for q in range(7-p)]
    
    for sig in signatures:
        L, _ = cf.Cl(*sig)
        # Random multivector with 64 coefficients
        A = L.MultiVector(np.random.uniform(-1, 1, 64))
        
        try:
            A_inv = algorithm(A)
            if A_inv:
                # Calculate the product A * A_inv
                check = A * A_inv
                
                # Create the target identity array (1.0 at index 0, 0.0 elsewhere)
                identity_val = np.zeros(64)
                identity_val[0] = 1.0
                
                # Calculate Mean Absolute Error (MAE)
                # np.abs(check.value - identity_val) gives the error for each of the 64 slots
                avg_error = np.mean(np.abs(check.value - identity_val))
                
                # Success threshold (still using 1e-5 for the PASS/FAIL flag)
                success = avg_error < 1e-5
                status = "✅ PASS" if success else "❌ FAIL"
                error_str = f"{avg_error:.2e}"
            else:
                status = "⚠️  SINGULAR"
                error_str = "N/A"
        except Exception:
            status = "🔥 ERROR"
            error_str = "N/A"
            
        print(f"{str(sig):<18} | {status:<10} | {error_str:<20}")

# ======================================

if __name__ == "__main__":
    # these are inverse functions built into 'clifford' 
    run_tests(shirokov_inverse)
    run_tests(perwass_inverse)
    # these are alternative functions for 6D
    run_tests(jones_6D_inverse)
    run_tests(acus_6D_inverse)


Testing shirokov_inverse
Signature (p,q,r)  | Status     | Avg Abs Error (MAE) 
-------------------------------------------------------
(0, 0, 6)          | ✅ PASS     | 1.85e-14            
(0, 1, 5)          | ✅ PASS     | 4.72e-14            
(0, 2, 4)          | ✅ PASS     | 2.55e-15            
(0, 3, 3)          | ✅ PASS     | 1.97e-15            
(0, 4, 2)          | ✅ PASS     | 1.13e-15            
(0, 5, 1)          | ✅ PASS     | 4.80e-15            
(0, 6, 0)          | ✅ PASS     | 2.65e-15            
(1, 0, 5)          | ✅ PASS     | 1.27e-13            
(1, 1, 4)          | ✅ PASS     | 1.49e-07            
(1, 2, 3)          | ✅ PASS     | 4.29e-15            
(1, 3, 2)          | ✅ PASS     | 1.02e-15            
(1, 4, 1)          | ✅ PASS     | 5.24e-16            
(1, 5, 0)          | ✅ PASS     | 7.26e-16            
(2, 0, 4)          | ✅ PASS     | 6.83e-12            
(2, 1, 3)          | ✅ PASS     | 3.33e-13            
(2, 2, 2)          | ✅ PASS     | 6.31